# **RAG pipeline**
- nootbook to do all RAG pipeline

### **Query** -> **embedding** -> **search_vector_store** -> **Retrieval top k result** -> **send context and query to llm**

## import the dependencies

In [95]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

import os
from pathlib import Path
import langdetect
from dotenv import load_dotenv

load_dotenv()

ar_vector_store_path = Path("../data/vector_store_ar")
en_vector_store_path = Path("../data/vector_store_en")

## Load Vector Store

### Load Embeddings

In [96]:
def load_embeddings(model_name):
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    return embeddings

embeddings = load_embeddings("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
print("✅ Embedding model loaded")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3408.74it/s]


✅ Embedding model loaded


### Load Vector Store

In [97]:
def load_vector_store(path,embeddings):
    vector_store = FAISS.load_local(path, embeddings, allow_dangerous_deserialization=True)
    return vector_store

ar_vector_store = load_vector_store(ar_vector_store_path, embeddings)
ar_retrieval = ar_vector_store.as_retriever(search_kwargs={"k": 3})
print(f"✅ arabic Vector store loaded from {ar_vector_store_path}")

en_vector_store = load_vector_store(en_vector_store_path, embeddings)
en_retrieval = en_vector_store.as_retriever(search_kwargs={"k": 3})
print(f"✅ English Vector store loaded from {en_vector_store_path}")

✅ arabic Vector store loaded from ..\data\vector_store_ar
✅ English Vector store loaded from ..\data\vector_store_en


## LLM

In [98]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature = 0,
)

print("✅ LLM loaded")

✅ LLM loaded


### prompt

In [99]:
prompt = ChatPromptTemplate.from_template(
   """
    You are a legal assistant specialized in the Egyptian Civil Code.

    Follow these rules strictly:

    1. Answer the question using ONLY the legal articles provided in the context.
    2. Do NOT use external legal knowledge or information that is not explicitly stated in the context.
    3. Mention the relevant article number(s) in your answer.
    4. Answer in the SAME LANGUAGE as the user's question:
    - If the question is in Arabic, answer in Arabic.
    - If the question is in English, answer in English.
    5. Do NOT translate the answer into another language.
    6. If the provided articles do not contain enough information to answer the question, say:
    "No sufficient information found in the Civil Code."
    7. Do not make general legal conclusions from an article that applies only to a specific situation.
    8. If the retrieved articles are not relevant to the question, explicitly say that the provided articles are not sufficient to answer the question.

    Legal Articles:
    {context}

    Question:
    {input}

    Answer:
"""
)


In [100]:
# Chain
def format_docs(docs):
    return "\n\n".join([
        f"Article {doc.metadata['article_number']}:\n{doc.page_content}"
        for doc in docs
    ])

def get_chain(lang):
    if lang == "ar":
        retrieval = ar_retrieval
    elif lang == "en":
        retrieval = en_retrieval
    else:
        raise ValueError("Unsupported language")

    chain = (
            {"context": retrieval | format_docs, "input": RunnablePassthrough()}
            | prompt
            | llm
            | StrOutputParser()
        )
    return chain

retreival_chain_ar = get_chain("ar")
retreival_chain_en = get_chain("en")


## TEST

In [101]:
def ask_legal_question(question : str) -> dict:

    # detect the question language
    lang = langdetect.detect(question)

    # get docs from the retriever
    if lang == "ar":
        retrieval = ar_retrieval
        chain = retreival_chain_ar
    elif lang == "en":
        retrieval = en_retrieval
        chain = retreival_chain_en
    else:
        raise ValueError("Unsupported language")

    docs = retrieval.invoke(question)
    for doc in docs:
        print(f"=== Article {doc.metadata['article_number']} ===")
        print(doc.page_content)
        print()

    
    # chain 
    answer = chain.invoke(question)
    
    # sources 
    sources = list(set([
        f"Article {doc.metadata['article_number']}"
        for doc in docs
    ]))
    
    return {
        "question": question,
        "answer":   answer,
        "sources":  sources
    }

### Arabic Test

In [102]:
response = ask_legal_question("ما هي شروط العقد؟")
print(f"Question: {response['question']}")
print(f"Answer: {response['answer']}")
print(f"Sources: {response['sources']}")

=== Article 201 ===
مادة 201
إذا وعد شخص بإبرام عقد ثم نكل وقاضاه المتعاقد الآخر طالباً تنفيذ الوعد ، وكانت الشروط اللازمة لتمام العقد وبخاصة ما يتعلق منها بالشكل متوافرة ، قام الحكم متى حاز قوة الشيء المقضي به مقام العقد .

=== Article 841 ===
مادة 841
١  (يجب تنفيذ العقد طبقا لما اشتمل عليه وبطريقة تتفق مع ما يوجبه حسن النية. ( ٢  (ولا يقتصر العقد على إلزام المتعاقد بما ورد فيه.

=== Article 101 ===
مادة 101
(١ (الاتفاق الذي يعد بموجبه كلا المتعاقدين أو أحدهما بابرام عقد معين في المستقبل لا ينعقد، إلا إذا عينت جميع المسائل الجوهرية للعقد المراد إبرامه، والمدة التي يجب إبرامه فيها . (٢ (إذا اشترط القانون ل تمام العقد استيفاء شكل معين، فهذا الشكل تجب مراعاته أيضاً في الاتفاق الذي يتضمن الوعد بإبرام هذا العقد .

Question: ما هي شروط العقد؟
Answer: وفقاً للمادة 101، لا ينعقد الاتفاق إلا إذا عينت جميع المسائل الجوهرية للعقد المراد إبرامه والمدة التي يجب إبرامه فيها، وإذا كان القانون يفرض شكلًا معينًا، يجب مراعاته. كما تنص المادة 201 على أن الحكم يُعطي قوة الشيء المقضي به إذا كانت الشروط ا

### English Test

In [103]:
response = ask_legal_question("What are the conditions of a valid contract?")
print(f"Question: {response['question']}")
print(f"Answer: {response['answer']}")
print(f"Sources: {response['sources']}")

=== Article 743 ===
Article 743
Subject to the requirements of the law as to the form of contracts for gifts, a contract providing for an annuity is valid only if made in writing.

=== Article 144 ===
Article 144
When a void or voidable contract contains the elements of another contract, the contract will be deemed to be valid to the extent of the other contract, if it appears that the parties intended to conclude such another contract.

=== Article 101 ===
Article 101
An agreement by which the two parties, or one of them, promise to enter into a particular contract in the future, is only binding if all the essential points of the contract envisaged and the time when the contract should be concluded are stated. When the law provides that a contract shall not be valid unless a certain form is observed, this form must also be observed in any agreement embodying a promise to enter into such a contract.

Question: What are the conditions of a valid contract?
Answer: The Civil Code states t